# Data Preparation

사출성형 품질 분석 팀 프로젝트의 기존 과제 노트북입니다.
팀: 락스메이, 정유진, 모티타.

**게시용 정리:** 원 분석 코드는 유지하고 파일명·데이터 경로를 정리했습니다.
이 노트북의 표와 출력은 기존 실행 기록이며, 게시 시점에 다시 실행한 결과가 아닙니다.
대표 그림은 `results/figures/`로 분리했습니다. 원본에 없던 실험 결과를 추가하지 않았습니다.

**라벨:** 최종 모델에서 `1=정상(Y)`, `0=불량(N)`입니다. 기본 precision/recall/F1은 정상 클래스 기준입니다.
문서에 남아 있는 defect/positive 설명은 이 라벨 정의와 함께 읽어야 합니다.
성능 해석과 전처리 한계는 [실험 기록](../docs/experiment-notes.md)을 확인하세요.

저장소 루트에서 `jupyter lab`을 실행하고 이 파일을 열어 순서대로 실행하세요.


In [ ]:
from pathlib import Path
Path("../outputs").mkdir(exist_ok=True)


# Data Quality and Preparation

**Dataset:** KAMP Injection Molding AI Dataset (labeled & unlabeled variants)  
**Team 4:** 락스메이, 정유진, 모티타  
**Course:** data analytics for electronic manufacturing  / 2nd Semester

In [ ]:
# ==============================================================
# 0. Imports & Load Data
# ==============================================================
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("default")
sns.set_theme(style="whitegrid", palette="pastel")

df = pd.read_csv("../data/labeled_data.csv")
print("Shape:", df.shape)
print(df.dtypes.head())
display(df.head(3))

print("\nRemark: Data loaded successfully —", df.shape[0], "samples and",
      df.shape[1], "features. Target variable is 'PassOrFail'.")

Shape: (7996, 45)
_id                    object
TimeStamp              object
PART_FACT_PLAN_DATE    object
PART_FACT_SERIAL        int64
PART_NAME              object
dtype: object


                        _id            TimeStamp     PART_FACT_PLAN_DATE  \
0  5f8928bb9c0189cc666ef19b  2020-10-16 04:57:47  2020-10-16 오전 12:00:00   
1  5f8928de9c0189cc666ef20b  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   
2  5f8928df9c0189cc666ef213  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   

   PART_FACT_SERIAL              PART_NAME EQUIP_CD  EQUIP_NAME PassOrFail  \
0                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기          Y   
1                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기          Y   
2                23  CN7 W/S SIDE MLD'G LH      S14  650톤-우진2호기          Y   

  Reason  Injection_Time  ...  Mold_Temperature_3  Mold_Temperature_4  \
0    NaN            9.59  ...           24.799999                27.5   
1    NaN            9.60  ...           24.799999                27.6   
2    NaN            9.60  ...           24.799999                27.6   

   Mold_Temperature_5  Mold_Temperature_6  Mold_Temperature_7  \
0                 0.0   


Remark: Data loaded successfully — 7996 samples and 45 features. Target variable is 'PassOrFail'.


## 1. Data Overview & Target Distribution

In [ ]:
# Info + descriptive stats
df.info()
desc = df.describe().T
display(desc.head(15))

# Target distribution
plt.figure(figsize=(5, 3))
sns.countplot(x="PassOrFail", data=df, palette="Set2")
plt.title("Target Distribution: Pass vs Fail")
plt.xlabel("PassOrFail")
plt.ylabel("Count")
plt.show()

print("Remark: Dataset is imbalanced — majority of parts are Pass (Y), only a small portion are Fail (N).")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7996 entries, 0 to 7995
Data columns (total 45 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   _id                       7996 non-null   object 
 1   TimeStamp                 7996 non-null   object 
 2   PART_FACT_PLAN_DATE       7996 non-null   object 
 3   PART_FACT_SERIAL          7996 non-null   int64  
 4   PART_NAME                 7996 non-null   object 
 5   EQUIP_CD                  7996 non-null   object 
 6   EQUIP_NAME                7996 non-null   object 
 7   PassOrFail                7996 non-null   object 
 8   Reason                    71 non-null     object 
 9   Injection_Time            7996 non-null   float64
 10  Filling_Time              7996 non-null   float64
 11  Plasticizing_Time         7996 non-null   float64
 12  Cycle_Time                7996 non-null   float64
 13  Clamp_Close_Time          7996 non-null   float64
 14  Cushion_

                           count        mean         std         min  \
PART_FACT_SERIAL          7996.0   11.248749    5.005191    2.000000   
Injection_Time            7996.0    8.242189    3.107724    1.050000   
Filling_Time              7996.0    3.896309    1.284451    0.930000   
Plasticizing_Time         7996.0   16.209505    1.443720   10.920000   
Cycle_Time                7996.0   59.900902    0.885680   58.840000   
Clamp_Close_Time          7996.0    7.064088    0.136595    5.160000   
Cushion_Position          7996.0  653.408195   10.165106   11.100000   
Switch_Over_Position      7996.0    0.330320   14.654889    0.000000   
Plasticizing_Position     7996.0   66.054484    5.418233   53.549999   
Clamp_Open_Position       7996.0  544.244111  236.351404    4.630000   
Max_Injection_Speed       7996.0   66.923612   26.460186   22.299999   
Max_Screw_RPM             7996.0   30.701989    0.159179   27.100000   
Average_Screw_RPM         7996.0  149.023424  130.828781   25.50

/tmp/ipython-input-532425377.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x="PassOrFail", data=df, palette="Set2")


[Historical plot output omitted; rerun this cell to display it.]

Remark: Dataset is imbalanced — majority of parts are Pass (Y), only a small portion are Fail (N).


## 2. Missing Data Analysis & MCAR/MAR/MNAR Reasoning

In [ ]:
# Missing percentage per column
missing = df.isna().mean().sort_values(ascending=False) * 100
display(missing.to_frame("Missing_%").head(20))

# Missingness map
plt.figure(figsize=(10, 4))
sns.heatmap(df.isna(), cbar=False)
plt.title("Missingness Map (NaN-based)")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.show()

print("Remark: There are effectively no NaN-based missing values in the dataset.")

# Reason vs PassOrFail
crosstab_reason = pd.crosstab(df["PassOrFail"], df["Reason"])
display(crosstab_reason)

print('Observation: "Reason" is "None" for all Pass (Y) parts, and a specific '
      'failure reason for Fail (N) parts. This is structural missingness (not random).')

# Clean Reason
df["Reason_clean"] = df["Reason"].replace({"None": "NoDefectReason"})
print(df["Reason_clean"].value_counts())

mcar_mar_mnar = """
MCAR (Missing Completely At Random): missingness unrelated to any data.
MAR  (Missing At Random): missingness depends only on observed variables.
MNAR (Missing Not At Random): missingness depends on the missing value itself.

In this dataset, numeric columns have 0% NaNs. "Reason" is structurally
missing for good parts (Pass), not randomly. It is closer to "not applicable"
than true MCAR/MAR/MNAR. Therefore, no numeric imputation is required.
We keep all rows and simply relabel "None" as "NoDefectReason".
"""
print(mcar_mar_mnar)

                       Missing_%
Reason                 99.112056
TimeStamp               0.000000
PART_FACT_PLAN_DATE     0.000000
PART_FACT_SERIAL        0.000000
_id                     0.000000
PART_NAME               0.000000
EQUIP_CD                0.000000
EQUIP_NAME              0.000000
PassOrFail              0.000000
Injection_Time          0.000000
Filling_Time            0.000000
Plasticizing_Time       0.000000
Cycle_Time              0.000000
Clamp_Close_Time        0.000000
Cushion_Position        0.000000
Switch_Over_Position    0.000000
Plasticizing_Position   0.000000
Clamp_Open_Position     0.000000
Max_Injection_Speed     0.000000
Max_Screw_RPM           0.000000

[Historical plot output omitted; rerun this cell to display it.]

Remark: There are effectively no NaN-based missing values in the dataset.


Reason      가스  미성형  초기허용불량
PassOrFail                 
N           35   16      20

Observation: "Reason" is "None" for all Pass (Y) parts, and a specific failure reason for Fail (N) parts. This is structural missingness (not random).
Reason_clean
가스        35
초기허용불량    20
미성형       16
Name: count, dtype: int64

MCAR (Missing Completely At Random): missingness unrelated to any data.
MAR  (Missing At Random): missingness depends only on observed variables.
MNAR (Missing Not At Random): missingness depends on the missing value itself.

In this dataset, numeric columns have 0% NaNs. "Reason" is structurally
missing for good parts (Pass), not randomly. It is closer to "not applicable"
than true MCAR/MAR/MNAR. Therefore, no numeric imputation is required.
We keep all rows and simply relabel "None" as "NoDefectReason".



## 3. Outlier Detection & Handling (Two Methods)

We apply **two outlier detection methods** on key process variables:
- `Injection_Time`
- `Filling_Time`
- `Average_Screw_RPM`
- `Switch_Over_Position`
- `Clamp_Open_Position`

**Methods used:**
1. **IQR method**: values outside `[Q1 − 1.5 × IQR, Q3 + 1.5 × IQR]` are considered outliers.
2. **Z-score method**: values with `|z| > 3` (three standard deviations from mean).

**Final handling decision:**
- We **keep all rows** (no row deletion), to avoid losing rare but potentially valid cycles.
- For modeling, we use **IQR-clipped versions** of these features (suffix `_clipped`).
- Z-score is used to **flag** extreme points (suffix `_z_outlier`), not to remove them.
- Original raw columns are kept for interpretation and audit.

In [ ]:
# Selected numeric columns for outlier analysis
num_cols = ["Injection_Time", "Filling_Time", "Average_Screw_RPM",
            "Switch_Over_Position", "Clamp_Open_Position"]

def iqr_clip(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return series.clip(lower, upper), lower, upper

def zscore_mask(series, threshold=3.0):
    z = (series - series.mean()) / series.std()
    return (np.abs(z) > threshold)

iqr_bounds = {}
outlier_summary = []

for col in num_cols:
    # IQR clipping
    clipped, low, high = iqr_clip(df[col])
    df[col + "_clipped"] = clipped
    iqr_bounds[col] = (low, high)

    # Z-score flags (True = extreme)
    z_mask = zscore_mask(df[col])
    df[col + "_z_outlier"] = z_mask.astype(int)

    # Summary
    n_iqr = int(((df[col] < low) | (df[col] > high)).sum())
    n_z = int(z_mask.sum())
    outlier_summary.append({
        "Feature": col,
        "IQR_lower": round(low, 3),
        "IQR_upper": round(high, 3),
        "IQR_outlier_count": n_iqr,
        "Z_outlier_count_(>|3σ|)": n_z
    })

outlier_table = pd.DataFrame(outlier_summary)
display(outlier_table)

print("Remark: IQR bounds define clipping thresholds; rows outside these bounds are not dropped, "
      "but their values are clipped into the valid range. Z-score flags mark extremely rare points.")

                Feature  IQR_lower  IQR_upper  IQR_outlier_count  \
0        Injection_Time      9.415      9.695               1322   
1          Filling_Time      4.295      4.575               1322   
2     Average_Screw_RPM   -365.600    687.200                  0   
3  Switch_Over_Position      0.000      0.000                  6   
4   Clamp_Open_Position    647.990    647.990               1296   

   Z_outlier_count_(>|3σ|)  
0                        0  
1                        4  
2                        0  
3                        4  
4                        0  

Remark: IQR bounds define clipping thresholds; rows outside these bounds are not dropped, but their values are clipped into the valid range. Z-score flags mark extremely rare points.


In [ ]:
# Boxplots before vs after clipping
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

sns.boxplot(data=df[num_cols], ax=axes[0], palette="Set3")
axes[0].set_title("Boxplots BEFORE Outlier Treatment")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(data=df[[c + "_clipped" for c in num_cols]], ax=axes[1], palette="Set2")
axes[1].set_title("Boxplots AFTER IQR Clipping")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print("Remark: After clipping, extreme tails are reduced while the bulk distribution remains similar, "
      "making models less sensitive to a few abnormal points.")

[Historical chart exported to results/figures/outlier-handling.png]

Remark: After clipping, extreme tails are reduced while the bulk distribution remains similar, making models less sensitive to a few abnormal points.


In [ ]:
# Example: Distribution before/after for Injection_Time
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df["Injection_Time"], bins=30, kde=True, ax=axes[0], color="salmon")
axes[0].set_title("Injection_Time (Raw)")

sns.histplot(df["Injection_Time_clipped"], bins=30, kde=True, ax=axes[1], color="skyblue")
axes[1].set_title("Injection_Time (IQR-clipped)")

plt.tight_layout()
plt.show()

print("Remark: Clipping slightly pulls in the extreme right tail but keeps the main mode and shape intact.")

[Historical plot output omitted; rerun this cell to display it.]

Remark: Clipping slightly pulls in the extreme right tail but keeps the main mode and shape intact.


### Outlier Handling – Final Decision

- **We do NOT drop any rows** – to avoid losing rare but possibly meaningful production conditions.
- For the selected process variables, we create **clipped versions** (e.g. `Injection_Time_clipped`) using the IQR rule.
- We also create **binary Z-score flags** (e.g. `Injection_Time_z_outlier`) to indicate extremely unusual cycles.
- **Modeling recommendation:** use `_clipped` variables as numeric inputs. The `_z_outlier` flags can be used
  as additional binary features or for further domain investigation.
- Original raw columns are kept for interpretability and traceability.

## 4. Numerical Transformations (4+ Variables)

In [ ]:
# Log transform for skewed time feature
df["Injection_Time_log"] = np.log1p(df["Injection_Time_clipped"])

# Square-root transform for Filling_Time
df["Filling_Time_sqrt"] = np.sqrt(df["Filling_Time_clipped"])

# Z-score standardization for Max_Injection_Speed
mu_speed = df["Max_Injection_Speed"].mean()
std_speed = df["Max_Injection_Speed"].std()
df["Max_Injection_Speed_z"] = (df["Max_Injection_Speed"] - mu_speed) / std_speed

# Robust scaling for Average_Screw_RPM_clipped
median_rpm = df["Average_Screw_RPM_clipped"].median()
iqr_rpm = df["Average_Screw_RPM_clipped"].quantile(0.75) - df["Average_Screw_RPM_clipped"].quantile(0.25)
df["Average_Screw_RPM_robust"] = (df["Average_Screw_RPM_clipped"] - median_rpm) / iqr_rpm

display(df[[
    "Injection_Time_clipped", "Injection_Time_log",
    "Filling_Time_clipped", "Filling_Time_sqrt",
    "Max_Injection_Speed", "Max_Injection_Speed_z",
    "Average_Screw_RPM_clipped", "Average_Screw_RPM_robust"
]].head())

print("Remark: We applied four different numeric transformations to different variables "
      "to reduce skewness and put them on comparable scales.")

   Injection_Time_clipped  Injection_Time_log  Filling_Time_clipped  \
0                    9.59            2.359910                  4.47   
1                    9.60            2.360854                  4.48   
2                    9.60            2.360854                  4.48   
3                    9.59            2.359910                  4.48   
4                    9.59            2.359910                  4.48   

   Filling_Time_sqrt  Max_Injection_Speed  Max_Injection_Speed_z  \
0           2.114237            55.400002              -0.435508   
1           2.116601            55.299999              -0.439287   
2           2.116601            55.299999              -0.439287   
3           2.116601            55.299999              -0.439287   
4           2.116601            55.299999              -0.439287   

   Average_Screw_RPM_clipped  Average_Screw_RPM_robust  
0                      292.5                   1.00038  
1                      292.5                   1.0

Remark: We applied four different numeric transformations to different variables to reduce skewness and put them on comparable scales.


In [ ]:
# Visual check: Injection_Time before vs log-transformed
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df["Injection_Time_clipped"], bins=30, kde=True, ax=axes[0], color="salmon")
axes[0].set_title("Injection_Time_clipped")

sns.histplot(df["Injection_Time_log"], bins=30, kde=True, ax=axes[1], color="skyblue")
axes[1].set_title("Injection_Time_log")

plt.tight_layout()
plt.show()

print("Remark: Log transformation makes the distribution more symmetric and less heavy-tailed.")

[Historical chart exported to results/figures/numeric-transformation.png]

Remark: Log transformation makes the distribution more symmetric and less heavy-tailed.


## 5. Categorical Encoding (4 Variables)

In [ ]:
# Binary encode PassOrFail
df["PassOrFail_bin"] = df["PassOrFail"].map({"Y": 1, "N": 0})
display(df[["PassOrFail", "PassOrFail_bin"]].head())
print('Remark: "PassOrFail" is encoded as 1=Pass, 0=Fail for modeling.')

# One-hot encode Reason_clean, EQUIP_NAME, PART_NAME
cat_cols = ["Reason_clean", "EQUIP_NAME", "PART_NAME"]
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=False)

# Frequency encoding for EQUIP_CD
freq_map = df["EQUIP_CD"].value_counts(normalize=True)
df_enc["EQUIP_CD_freq"] = df["EQUIP_CD"].map(freq_map)

print("Sample of one-hot encoded Reason_clean columns:")
print([c for c in df_enc.columns if c.startswith("Reason_clean_")][:5])
print("Remark: We encoded 4 categorical variables: PassOrFail (binary), Reason_clean, "
      "EQUIP_NAME, PART_NAME (one-hot), plus EQUIP_CD (frequency).")

  PassOrFail  PassOrFail_bin
0          Y               1
1          Y               1
2          Y               1
3          Y               1
4          Y               1

Remark: "PassOrFail" is encoded as 1=Pass, 0=Fail for modeling.
Sample of one-hot encoded Reason_clean columns:
['Reason_clean_가스', 'Reason_clean_미성형', 'Reason_clean_초기허용불량']
Remark: We encoded 4 categorical variables: PassOrFail (binary), Reason_clean, EQUIP_NAME, PART_NAME (one-hot), plus EQUIP_CD (frequency).


## 6. Final Prepared Dataset (Ready for Modeling)

In [ ]:
drop_raw = ["PassOrFail", "Reason", "Reason_clean", "EQUIP_CD", "EQUIP_NAME", "PART_NAME"]
drop_raw = [c for c in drop_raw if c in df_enc.columns]

X = df_enc.drop(columns=drop_raw + ["PassOrFail_bin"])
y = df_enc["PassOrFail_bin"]

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)

X.to_csv("../outputs/prepared_features.csv", index=False)
y.to_csv("../outputs/prepared_target.csv", index=False)

print("Prepared features saved to prepared_features.csv")
print("Prepared target saved to prepared_target.csv")
print("Remark: X contains cleaned numeric features (with outlier clipping and transforms) "
      "plus encoded categorical features. y is the binary quality label.")

Final X shape: (7996, 67)
Final y shape: (7996,)
Prepared features saved to prepared_features.csv
Prepared target saved to prepared_target.csv
Remark: X contains cleaned numeric features (with outlier clipping and transforms) plus encoded categorical features. y is the binary quality label.


## 7. Summary of Data Quality Decisions

| Aspect            | Method / Action                                          | Justification |
|-------------------|---------------------------------------------------------|---------------|
| Missing Data      | No NaN imputation; structural "Reason" cleaned        | Avoids artificial values; data is essentially complete |
| Outliers          | IQR clipping + Z-score flagging, **no row deletion**    | Reduce influence of extremes while keeping rare cycles |
| Numeric Transforms| log, sqrt, z-score, robust scaling (4+ variables)       | Stabilize variance, reduce skew, normalize scales |
| Categorical       | Binary + one-hot + frequency encoding (4+ variables)    | Convert all categories into model-ready numeric form |
| Final Dataset     | Use `_clipped` vars + encoded categoricals as features  | Balanced between robustness and information preservation |

These steps provide a clear, well-justified data quality and preparation pipeline
that is ready for downstream modeling.